# UMR QB1 - Seminar zur Genexpressionsanalyse

---

## Medizinisches Problem & Forschungsfrage

> *Was sind die molekularen Unterschiede zwischen Krebszellen und gesundem menschlichem Gewebe, und wie können wir diese Unterschiede nutzen, um neue therapeutische Ziele und deren Medikamente für die Krebsbehandlung zu identifizieren?*

### Klinischer Hintergrund

- Krebsheterogenität: Verschiedene Krebsarten haben unterschiedliche molekulare Signaturen.
- Präzisionsmedizin: Behandlungen müssen auf die jeweilige Krebsart zugeschnitten sein.
- Medikamentenresistenz: Unter Behandlung entwickeln Krebszellen Resistenzen.

### Analyseziele

- Molekulare Krebssignaturen: Gene, die in Krebsgewebe gegenüber gesundem Gewebe konsistent verändert sind
- Therapeutische Schwachstellen: Signalwege, die sich mit Medikamenten gezielt ansteuern lassen
- Medikamentenumwidmung: bestehende Medikamente, die Krebs behandeln könnten
- Biomarker: Gene, die das Ansprechen auf die Behandlung vorhersagen könnten

## Lernziele

- Eine differentielle Expressionsanalyse durchführen, um Krebs-Biomarker zu identifizieren
- Eine Pathway-Anreicherungsanalyse durchführen, um die Krebsbiologie zu verstehen
- Computergestützte Medikamentenumwidmung für die Suche nach Therapien einsetzen

---


## Setup und Einführung

### Conda installieren

Die Zelle startet die Laufzeit nach der Installation automatisch neu. Warten Sie, bis der Neustart abgeschlossen ist, und fahren Sie dann fort. Ignorieren Sie dabei Fehlermeldungen und Warnungen.

In [ ]:
! pip install -q condacolab
import condacolab
condacolab.install()

### Conda-Installation überprüfen

In [ ]:
import condacolab
condacolab.check()

### Benötigte R-Pakete installieren

In [ ]:
! conda install -c bioconda -c conda-forge bioconductor-deseq2 r-optparse r-ggplot2 r-gprofiler2 r-dplyr r-tidyr -y -q 2>&1

## Arbeitsumgebung: Notebooks, Colab und Installation

Dieses Notebook liegt auf GitHub und läuft auf einem Rechner von Google. Auf Ihrem eigenen Computer brauchen Sie nur einen Webbrowser.

![Schema: GitHub speichert Notebook, Skripte und Daten. Der Webbrowser öffnet das Notebook in Colab, Colab führt die Zellen auf einem temporären Linux-Rechner in der Google-Cloud aus, dieser lädt die Skripte und Daten per git clone und schickt Tabellen und Grafiken zurück an den Browser.](https://raw.githubusercontent.com/barrantesisrael/QB1seminar2026/main/colab_architektur.png)

Der Ablauf in vier Schritten:

1. Sie öffnen den Link zum Notebook auf GitHub, und Colab zeigt es im Browser an.
2. Wenn Sie Zellen ausführen, startet Colab dafür einen Linux-Rechner in der Google-Cloud.
3. Dieser Rechner lädt mit `git clone` die Skripte und Daten von GitHub.
4. Tabellen und Grafiken erscheinen unter den Zellen in Ihrem Browser.

### Jupyter-Notebook und Google Colab

Ein [Jupyter-Notebook](https://jupyter.org) ist ein Dokument aus Text, Code und Ergebnissen. Die Code-Zellen laufen einzeln, und das Ergebnis steht direkt darunter.

[Google Colab](https://colab.research.google.com) ist ein kostenloser Dienst, der Jupyter-Notebooks im Browser ausführt, ohne dass Sie etwas installieren. Der Google-Rechner ist temporär: Er wird nach einiger Zeit ohne Aktivität gelöscht, und Google garantiert weder Laufzeit noch Ausstattung. Die Setup-Zellen müssen deshalb in jeder neuen Sitzung erneut laufen.

### Software installieren: drei Wege

| Weg                      | Idee                                                  | Nachteil              |
|:-------------------------|:------------------------------------------------------|:----------------------|
| Direkt                   | Jedes Programm einzeln auf dem eigenen System         | Versionskonflikte     |
| Umgebungsmanager (conda) | Getrennte "Kisten" mit festgelegten Programmversionen | Etwas Einrichtung     |
| Container (Docker)       | Ganzes System samt Betriebssystem verpackt            | Groß und aufwendig    |

Im Seminar verwenden wir conda, installiert innerhalb von Colab (siehe Setup).

---

## Warum R, Python und die Kommandozeile?

In der Bioinformatik schreiben wir die Analyse als Code, statt sie mit der Maus zusammenzuklicken. Dafür brauchen wir eine Programmiersprache und die Kommandozeile.

### R und Python

- Beide Sprachen sind kostenlos (Open Source) und in der Forschung weltweit verbreitet.
- R wurde für Statistik und Grafiken entwickelt. Bioconductor bündelt Pakete für die Bioinformatik, darunter DESeq2, das wir gleich verwenden.
- Python ist eine allgemeine Programmiersprache. Sie eignet sich gut für Datenverarbeitung und für die Anbindung von Web-Diensten, zum Beispiel im Drug-Repositioning-Skript.
- Dieses Notebook ruft je ein R-Skript (DESeq2, Pathway-Analyse) und ein Python-Skript (Drug Repositioning) auf.
- Excel stößt bei großen Gentabellen schnell an Grenzen, und Mausklicks lassen sich nicht wiederholen. Ein Skript läuft bei jedem Aufruf gleich ab.
- KI-Assistenten (LLMs) erklären Code und helfen, Ergebnisse zu interpretieren, wie dieser [Beispiel-Chat mit Gemini](https://gemini.google.com/share/c4d4fe5d9be8) zeigt. Prüfen Sie die Antworten immer, bevor Sie sie übernehmen.

### Die Kommandozeile

Auf der Kommandozeile tippen Sie Befehle, statt Programme per Mausklick zu starten. Der Computer führt genau das aus, was dort steht.

- Viele Bioinformatik-Programme gibt es nur für die Kommandozeile, zum Beispiel Salmon, das unsere Count-Matrix erzeugt hat.
- Sequenzierdateien sind zu groß, um sie wie ein Dokument zu öffnen. Mit `head`, `wc` und `grep` prüfen Sie ihren Inhalt in Sekunden.
- Befehle lassen sich in einem Skript speichern und beliebig oft wiederholen.
- In Colab startet eine Zeile mit `!` einen Befehl, zum Beispiel `! git clone ...` weiter unten.

Referenz: Buffalo, V. (2015). Bioinformatics Data Skills. O'Reilly Media.

---

## Datensatz: Krebsproben vs. Kontrollproben

- Gegenüberstellung: Universal Human Reference (UHR, Krebszelllinien) vs. Human Brain Reference (HBR, gesundes menschliches Hirngewebe)
- Stichprobengröße: 6 Proben (3 Krebsreplikate, 3 Kontrollreplikate)
- Datentyp: Paired-End-RNA-Sequenzierung (Chromosom-22-Subset als Bildungsdatensatz)
- Referenz: Griffith M, Walker JR, Spies NC, Ainscough BJ, Griffith OL (2015) Informatics for RNA Sequencing: A Web Resource for Analysis on the Cloud. *PLoS Comput Biol* 11(8): e1004393. https://doi.org/10.1371/journal.pcbi.1004393

### Proben

#### UHR (Universal Human Reference): Krebsproben

- Zusammensetzung: Gesamt-RNA aus 10 verschiedenen menschlichen Krebszelllinien
- Enthaltene Krebsarten: Brust-, Leber-, Gebärmutterhals-, Hoden-, Gehirn- und Hautkrebs sowie Immunzellen (T-Zellen, B-Zellen, Makrophagen, Histiozyten)
- Klinische Relevanz: Die Mischung bildet gemeinsame molekulare Merkmale verschiedener Krebsarten ab und erleichtert die Suche nach therapeutischen Zielen, die mehreren Krebsarten gemeinsam sind.

#### HBR (Human Brain Reference): Kontrollproben

- Zusammensetzung: Gesamt-RNA aus den Gehirnen von 23 gesunden Kaukasiern, meist 60-80 Jahre alt
- Klinische Relevanz: Die Proben zeigen, wie die Genexpression in gesundem Gewebe aussieht, und dienen als Vergleichsbasis.

### Biologische Hypothese

Krebszellen zeigen im Vergleich zu gesundem Gewebe systematische Veränderungen der Genexpression. Erwartet werden:

1. Onkogene (krebsfördernde Gene), die überexprimiert sind
2. Tumorsuppressoren (krebsverhindernde Gene), die stillgelegt sind
3. Stoffwechselwege, die verändert sind und das Krebswachstum unterstützen

---

### Daten- und Skript-Repository

- Expressionsdaten: Vorverarbeitete RNA-seq-Datendateien und Count-Matrix
- Analyseskripte für jeden Schritt
- Referenzgenomdateien und Annotationen

![](https://github.com/barrantesisrael/QB1seminar2026/blob/main/getdata.png?raw=true)

In [ ]:
! git clone https://github.com/barrantesisrael/QB1seminar2026.git

In [ ]:
! ls

In [ ]:
# Change working directory permanently
import os
os.chdir('QB1seminar2026')

In [ ]:
! ls -l

In [ ]:
! bash getdata.sh

In [ ]:
! ls -l

---

## Genexpression

![Vergleich von qPCR, Microarrays und RNA-seq nach Sondenbedarf, Durchsatz und Nachweis neuer Transkripte](https://github.com/barrantesisrael/QB1seminar2026/blob/main/expression_technology_comparison_ed2.jpg?raw=true)

### RNA-seq

RNA-seq bestimmt für jedes Gen, wie viele RNA-Moleküle in einer Probe vorlagen. Dafür sind drei Schritte nötig:

1. Reads zuordnen: Bestimmen, von welchem Gen oder Transkript jeder Sequenzierungs-Read stammt
2. Zählen: Aufaddieren, wie viele Reads auf jedes Gen abgebildet werden
3. Normalisieren: Unterschiede in Sequenzierungstiefe und Genlänge ausgleichen

![Ablauf eines RNA-seq-Experiments von der Biopsie über Bibliotheksvorbereitung und Sequenzierung bis zur bioinformatischen Analyse](https://github.com/barrantesisrael/QB1seminar2026/blob/main/rnaseq_experiments.png?raw=true)

### Count-Daten

DESeq2 und andere count-basierte Modelle benötigen ganzzahlige rohe Counts, also die tatsächliche Anzahl der Sequenzierungs-Reads pro Gen.

![Read-Zuordnung zu den Exons eines Onkogens und eines Tumorsuppressors mit Read-Counts je Gewebe](https://github.com/barrantesisrael/QB1seminar2026/blob/main/rnaseq_mapping_example_ed2.jpg?raw=true)

![Expressionstabelle mit den Counts von zwölf Genen in je drei UHR- und HBR-Replikaten](https://github.com/barrantesisrael/QB1seminar2026/blob/main/rnaseq_expression_table_ed2.jpg?raw=true)

---

## 1 Daten

### 1.1 Datensatz: Metadaten und Sequenzierungen

Die Datei `samples.tsv` ordnet jede Probe einer Bedingung (`cancer` oder `normal`) zu und nennt Gewebetyp und klinischen Kontext.


In [ ]:
! cat samples.tsv

### 1.2 Genannotationen (GTF-Format)

- Spalte 1: Chromosomenname (22)
- Spalte 2: Annotationsquelle (z.B. Ensembl, HAVANA)
- Spalte 3: Feature-Typ (z.B. Gen, Transkript, Exon)
- Spalten 4-5: Start- und Endpositionen
- Spalte 7: Strang (+ oder -)
- Spalte 9: Attribute (gene_id, gene_name usw.)


In [ ]:
! tail chr22_with_ERCC92.gtf

### 1.3 DNA-Sequenz (FASTA-Format)

- Kopfzeile (`>`): Sequenzidentifikator und Beschreibung
- Sequenzzeilen: DNA-Basen (A, T, G, C)
- Verwendung: Referenzgenom für die Read-Zuordnung


In [ ]:
! head chr22_with_ERCC92.fa

### 1.4 Genexpressions-Count-Matrix

Die Count-Matrix ist die zentrale Datenstruktur jeder RNA-seq-Analyse:

- Zeilen: Gene oder Transkripte
- Spalten: Proben (UHR- und HBR-Replikate)
- Werte: Anzahl der Sequenzierungs-Reads, die dem jeweiligen Gen in der jeweiligen Probe zugeordnet sind

Die Matrix entsteht in drei Schritten:

1. Salmon quantifiziert die Reads pro Transkript.
2. Die Summe der Transkript-Counts eines Gens ergibt den Gen-Count.
3. Alle Proben stehen anschließend in einer gemeinsamen Matrix.


In [ ]:
! head counts.tsv

#### Aufbau der Count-Matrix

**Erste Spalte (Zeilennamen):**

- Transkript-Identifikatoren aus den Ensembl-Annotationen (z.B. ENST00000...), die den Einträgen der GTF-Datei entsprechen

**Proben-Spalten:**

- UHR_rep1, UHR_rep2, UHR_rep3: biologische Replikate gemischter Krebszelllinien (Krebsproben)
- HBR_rep1, HBR_rep2, HBR_rep3: biologische Replikate von gesundem Hirngewebe (Kontrollproben)

**Count-Werte:**

- Höhere Counts bedeuten höhere Expression.
- Null-Counts: Gen oder Transkript nicht exprimiert oder nicht nachweisbar
- Ähnliche Counts über Replikate hinweg zeigen zuverlässige Messungen.

**Medizinische Interpretation:**

- Hohe UHR-Counts weisen auf potenzielle Onkogene hin.
- Hohe HBR-Counts weisen auf potenzielle Tumorsuppressoren hin, die in Krebs stillgelegt sind.
- Ähnliche Counts in beiden Gruppen deuten auf Haushaltsgene hin, die nicht differenziell exprimiert sind.


---

## 2 Differentielle Expressionsanalyse

Die differentielle Expressionsanalyse identifiziert Gene, die zwischen zwei Bedingungen (hier Krebsproben vs. Kontrollproben) signifikant unterschiedlich exprimiert sind.

### Das [DESeq2](https://bioconductor.org/packages/release/bioc/html/DESeq2.html)-Verfahren

1. Normalisierung: Ausgleich von Unterschieden in der Bibliotheksgröße zwischen Proben
2. Dispersionsschätzung: Modellierung der Variabilität zwischen biologischen Replikaten
3. Statistisches Testen: Identifikation signifikant veränderter Gene
4. Multiple-Test-Korrektur: Anpassung der P-Werte für Tausende gleichzeitiger Tests



### Ausgabemetriken

- baseMean: durchschnittliche normalisierte Counts über alle Proben
- log2FoldChange: Stärke der Expressionsänderung (positiv = höher im Krebs)
- pvalue: statistische Signifikanz der Änderung
- padj: adjustierter P-Wert (nach Korrektur für multiple Tests)

### Signifikanzschwellen

- padj < 0,05: statistisch signifikant (FDR-korrigiert)
- |log2FC| > 1: biologisch bedeutsame Änderung (2-fach)
- Beide Kriterien zusammen: starke Kandidatengene

![Ablauf von run_deseq2.R: Count-Matrix und Probeninformation, Filtern, Normalisieren und Vergleichen mit DESeq2, Ergebnistabelle sowie PCA-Plot und Heatmap](https://github.com/barrantesisrael/QB1seminar2026/blob/main/deseq_flowchart.png?raw=true)

In [ ]:
! Rscript run_deseq2.R \
    --metadata samples.tsv \
    --expression counts.tsv \
    --gtf chr22_with_ERCC92.gtf \
    --output-degs degs.tsv \
    --output-plots pca_plot.png,heatmap.png

In [ ]:
! head degs.tsv

In [ ]:
! grep "ENST00000341184" chr22_with_ERCC92.gtf | cut -f9

### Datenvisualisierung zur biologischen Interpretation

Visualisierungen zeigen Muster über Tausende von Genen und decken Ausreißer und technische Probleme auf. `run_deseq2.R` erzeugt zwei Plots:

1. PCA Plot: Beziehungen zwischen Proben und Qualitätskontrolle
2. Heatmap: Expressionsmuster der 20 Gene mit dem kleinsten adjustierten P-Wert (z-Scores der VST-Counts)


In [ ]:
from IPython.display import Image, display
import os

display(Image('heatmap.png'))

Als Beispiel die Annotationen von `RPL3`:

In [ ]:
! grep "RPL3" chr22_with_ERCC92.gtf | cut -f9

---

## 3 Pathway-Anreicherungsanalyse

Die Pathway-Anreicherungsanalyse gruppiert Gene nach ihrer biologischen Funktion, statt einzelne Gene zu betrachten. Sie prüft, ob die krebsassoziierten Gene in bestimmten biologischen Prozessen überrepräsentiert sind und welche zellulären Signalwege bei Krebs dysreguliert sind. Beeinträchtigte Signalwege sind Ansatzpunkte für die Therapie, auch für Kombinationstherapien.


### Verwendete Datenbanken

1. [Gene Ontology](https://geneontology.org/) (GO), nur biologische Prozesse (BP): zelluläre Funktionen (z.B. Zellzyklus, Apoptose)
2. [KEGG](https://www.kegg.jp/): gut dokumentierte Signal- und Krankheitswege
3. [Reactome](https://reactome.org/): kuratierte biologische Wege und Reaktionen

### Erwartete Krebswege

- Hochreguliert: Zellzyklusfortschritt, DNA-Replikation, Metabolismus
- Herunterreguliert: Apoptose, DNA-Reparatur, Immunantwort

![Ablauf von pathway_analysis.R: signifikante Gene auswählen, in hoch- und herunterregulierte Gene aufteilen, Anreicherungsanalyse mit g:Profiler, Tabelle der angereicherten Signalwege](https://github.com/barrantesisrael/QB1seminar2026/blob/main/pathway_flowchart.png?raw=true)

In [ ]:
! Rscript pathway_analysis.R --input degs.tsv --output pathway_results.tsv

In [ ]:
! head pathway_results.tsv

---

## 4 Drug Repositioning

Drug Repositioning sucht neue therapeutische Anwendungen für bestehende Medikamente. Der Ansatz hat vier Vorteile:

1. Schnellere Entwicklung: 5-10 Jahre statt 15-20 Jahre bei neuen Medikamenten
2. Bekannte Sicherheitsprofile: Für bestehende Medikamente liegen Sicherheitsdaten vor.
3. Geringere Kosten: Risiko und Kosten der Medikamentenentwicklung sinken.
4. Schnelle klinische Anwendung: In einigen Fällen ist eine Off-Label-Verschreibung möglich.

Ein Beispiel ist Metformin: ursprünglich ein Diabetes-Medikament, wird es inzwischen in über 200 klinischen Studien zur Krebsprävention untersucht.

Das Skript konvertiert die DESeq2-Ergebnisse in ein Format, das mit L1000CDS2 kompatibel ist, einer großen Datenbank für Medikamentenumwidmung.


### L1000CDS2

[L1000CDS2](https://maayanlab.cloud/L1000CDS2/) (L1000 Characteristic Direction Signature)

- Datenbankumfang: Genexpressionssignaturen für >20.000 Medikamente, die an menschlichen Zelllinien getestet wurden
- LINCS-Programm: Teil der NIH-Initiative Library of Integrated Network-based Cellular Signatures

Der Algorithmus sucht Medikamente, die Krankheitssignaturen umkehren, in vier Schritten:

1. Eingangssignatur: die signifikant hoch- und herunterregulierten Gene (padj < 0,05, |log2FC| > 1; maximal 150 pro Richtung)
2. Datenbanksuche: Vergleich mit medikamenteninduzierten Expressionsveränderungen
3. Bewertung: Berechnung, wie gut jedes Medikament die Krebssignatur umkehrt
4. Rangfolge: Ausgabe der Medikamente, geordnet nach ihrem Potenzial gegen Krebs

![Ablauf von drug_repositioning.py: signifikante Gene auswählen, Krebssignatur bilden, L1000CDS2 nach umkehrenden Medikamenten durchsuchen, Tabelle der Medikamentenkandidaten](https://github.com/barrantesisrael/QB1seminar2026/blob/main/drugrepo_flowchart.png?raw=true)


In [ ]:
! python drug_repositioning.py --input degs.tsv --output drug_candidates.tsv

In [ ]:
! cat drug_candidates.tsv

### Interpretation der Drug-Repositioning-Ergebnisse

Die Ergebnisliste ordnet Verbindungen danach, wie gut sie Krebsgensignaturen umkehren. Für die [L1000CDS2-Scores](https://maayanlab.cloud/L1000CDS2/help/#result) gilt:

- Score: Überlappung zwischen der Eingabesignatur und der Medikamentensignatur (Anteil der Eingabegene)
- Höhere Scores: bessere Übereinstimmung mit der umgekehrten Krebssignatur und damit höheres therapeutisches Potenzial

### Validierungskategorien

**Bekannte Krebsmedikamente (positive Kontrollen):**

- Beispiele: Doxorubicin, Paclitaxel, Cisplatin, Tamoxifen, Imatinib
- Erscheinen sie in der Rangliste, stützt das den Ansatz: Die Krebssignatur ist biologisch relevant, und die Methode findet etablierte Krebstherapeutika.

**Vielversprechende Umwidmungskandidaten:**

- Statine (Cholesterinmedikamente): In Studien zeigten sich Anti-Krebs-Eigenschaften.
- Rapamycin (Immunsuppressivum): Gegenstand aktiver Krebs- und Altersforschung
- Chloroquin (Antimalariamittel): wird für Krebsanwendungen untersucht
- Alle drei Kandidaten erfordern weitere Validierung.

---

### Biologische Begründung

- Pathway-Targeting: Viele Krankheiten teilen molekulare Signalwege.
- Polypharmakologie: Einzelne Medikamente wirken oft auf mehrere biologische Ziele.
- Netzwerkeffekte: Medikamente können miteinander verbundene zelluläre Systeme beeinflussen.
- Zufallsfunde: Medikamente zeigen teils unerwartete positive Effekte bei anderen Krankheiten.



---

## Kontakt

Dr. rer. nat. Israel Barrantes <br>
Forschungsgruppe Translational Bioinformatics (Leiter)<br>
Institut für Biostatistik und Informatik in Medizin und Alternsforschung<br>
Universitätsmedizin Rostock<br>

E-Mail: israel.barrantes[bei]uni-rostock.de

---
Update 2026/10/26
